# Data Cleaning -- Training Dataset
**This notebook prepares Flatiron Health data for patients with advanced non–small cell lung cancer in anticipation of training a gradient-boosted survival model to predict mortality from the start of first-line treatment. Specifically, it processes and cleans the training cohort. Prior to data cleaning, the dataset is randomly split into training (80%) and testing (20%) subsets.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorNSCLC, merge_dataframes

## Split into train and test sets

In [2]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [3]:
df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [4]:
df.shape

(83095, 2)

In [5]:
processor = DataProcessorNSCLC()

In [6]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvNSCLC_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvNSCLC_Progression.csv',
                                           drop_dates = False)

2026-03-16 14:06:23,625 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (84881, 2) and unique PatientIDs: 84881
2026-03-16 14:06:23,697 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (83095, 3) and unique PatientIDs: 83095
2026-03-16 14:06:28,281 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 14:06:28,336 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (83095, 6) and unique PatientIDs: 83095. There are 0 out of 83095 patients with missing duration values


In [7]:
df = pd.merge(df, mortality_df[['PatientID', 'event']], on = 'PatientID', how = 'left')

In [8]:
df.shape

(83095, 3)

In [9]:
# Stratify split on event 
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['event'],  
    random_state = 42
)

In [10]:
train.shape

(66476, 3)

In [11]:
test.shape

(16619, 3)

In [12]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

## Clean CSV files 

In [13]:
train_ids = pd.read_csv('../outputs/train_patient_ids.csv')

In [14]:
train_ids = train_ids.PatientID.to_list()

In [15]:
index_date_df = df[df.PatientID.isin(train_ids)]

In [16]:
index_date_df.shape

(66476, 3)

In [17]:
index_date_df = index_date_df[['PatientID', 'StartDate']]

### Process Enhanced_AdvancedNSCLC.csv

In [18]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvancedNSCLC.csv',
                                         patient_ids = train_ids,
                                         drop_dates = False)

2026-03-16 14:06:28,697 - INFO - Successfully read Enhanced_AdvancedNSCLC.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-03-16 14:06:28,697 - INFO - Filtering for 66476 specific PatientIDs
2026-03-16 14:06:28,718 - INFO - Successfully filtered Enhanced_AdvancedNSCLC.csv file with shape: (66476, 6) and unique PatientIDs: 66476
2026-03-16 14:06:28,764 - INFO - Successfully processed Enhanced_AdvancedNSCLC.csv file with final shape: (66476, 8) and unique PatientIDs: 66476


In [19]:
index_date_df['StartDate'] = pd.to_datetime(index_date_df['StartDate'])

enhanced_df = pd.merge(enhanced_df, index_date_df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')
enhanced_df['days_adv_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [20]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [21]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         41313
III        14713
I           5681
II          3106
unknown     1654
0              9
Name: count, dtype: int64

In [22]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV given most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [23]:
enhanced_df.Histology.value_counts(dropna = False)

Histology
Non-squamous cell carcinoma    47364
Squamous cell carcinoma        16303
NSCLC histology NOS             2808
NaN                                1
Name: count, dtype: int64

In [24]:
enhanced_df['Histology'] = enhanced_df['Histology'].fillna('Non-squamous cell carcinoma')

In [25]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [26]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2026-03-16 14:06:28,901 - INFO - Successfully read Demographics.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-03-16 14:06:28,978 - INFO - Successfully processed Demographics.csv file with final shape: (66476, 6) and unique PatientIDs: 66476


In [27]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      34016
F      32456
NaN        4
Name: count, dtype: int64

In [28]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [29]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvNSCLCBiomarkers.csv

In [30]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-16 14:06:30,064 - INFO - Successfully read Enhanced_AdvNSCLCBiomarkers.csv file with shape: (1021678, 18) and unique PatientIDs: 93715
2026-03-16 14:06:30,567 - INFO - Successfully merged Enhanced_AdvNSCLCBiomarkers.csv df with index_date_df resulting in shape: (678715, 19) and unique PatientIDs: 57391
2026-03-16 14:06:33,448 - INFO - Successfully processed Enhanced_AdvNSCLCBiomarkers.csv file with final shape: (66476, 11) and unique PatientIDs: 66476


In [31]:
biomarkers_df['PDL1_percent_staining'] = biomarkers_df["PDL1_percent_staining"].map({
    '0%': '0%', 
    '<1%': '1-49%',
    '1%': '1-49%',
    '2% - 4%': '1-49%',
    '5% - 9%': '1-49%', 
    '10% - 19%': '1-49%',
    '20% - 29%': '1-49%',
    '30% - 39%': '1-49%',
    '40% - 49%': '1-49%',
    '50% - 59%': '>=50%', 
    '60% - 69%': '>=50%', 
    '70% - 79%': '>=50%',
    '80% - 89%': '>=50%', 
    '90% - 99%': '>=50%', 
    '100%': '>=50%',
})

biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].fillna('unknown')
biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].astype('category')

In [32]:
for biomarker in biomarkers_df.columns.drop('PDL1_percent_staining'):
    biomarkers_df[biomarker] = biomarkers_df[biomarker].fillna('unknown')
    biomarkers_df[biomarker].value_counts(dropna = False)

### Process ECOG.csv

In [33]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-16 14:06:34,061 - INFO - Successfully read ECOG.csv file with shape: (1623197, 4) and unique PatientIDs: 85577
2026-03-16 14:06:34,416 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (1188968, 5) and unique PatientIDs: 53530
2026-03-16 14:06:35,260 - INFO - Successfully processed ECOG.csv file with final shape: (66476, 3) and unique PatientIDs: 66476


In [34]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    22415
1      20744
0      13940
2       7498
3       1768
4        111
Name: count, dtype: int64

In [35]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [36]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [37]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-16 14:07:43,515 - INFO - Successfully read Vitals.csv file with shape: (29862758, 16) and unique PatientIDs: 114285
2026-03-16 14:07:58,029 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (20401661, 17) and unique PatientIDs: 66437
2026-03-16 14:08:07,393 - INFO - Successfully processed Vitals.csv file with final shape: (66476, 8) and unique PatientIDs: 66476


### Process Lab.csv

In [38]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-16 14:10:01,604 - INFO - Successfully read Lab.csv file with shape: (82093498, 17) and unique PatientIDs: 108761
2026-03-16 14:10:59,470 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (58193666, 18) and unique PatientIDs: 65528
2026-03-16 14:13:04,915 - INFO - Successfully processed Lab.csv file with final shape: (66476, 76) and unique PatientIDs: 66476


### Process MedicationAdministration.csv

In [39]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-16 14:13:14,712 - INFO - Successfully read MedicationAdministration.csv file with shape: (7486734, 11) and unique PatientIDs: 89605
2026-03-16 14:13:17,410 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (5104300, 12) and unique PatientIDs: 61459
2026-03-16 14:13:17,745 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (66476, 9) and unique PatientIDs: 66476


### Process Diagnosis.csv

In [40]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-16 14:13:23,390 - INFO - Successfully read Diagnosis.csv file with shape: (9083016, 6) and unique PatientIDs: 114756
2026-03-16 14:13:25,277 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (5755247, 7) and unique PatientIDs: 66476
2026-03-16 14:13:37,323 - INFO - Successfully processed Diagnosis.csv file with final shape: (66476, 39) and unique PatientIDs: 66476


### Process Enhanced_Mortality_V2.csv

In [41]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvNSCLC_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvNSCLC_Progression.csv',
                                           drop_dates = False)

2026-03-16 14:13:37,465 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (84881, 2) and unique PatientIDs: 84881
2026-03-16 14:13:37,530 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (66476, 3) and unique PatientIDs: 66476
2026-03-16 14:13:41,802 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 14:13:41,838 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (66476, 6) and unique PatientIDs: 66476. There are 0 out of 66476 patients with missing duration values


In [42]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [43]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [44]:
training_df = merge_dataframes(enhanced_df,
                               demographics_df,
                               biomarkers_df,
                               ecog_df,
                               vitals_df,
                               labs_df,
                               medications_df,
                               diagnosis_df,
                               mortality_df,
                               merge_type = 'inner')

2026-03-16 14:13:41,976 - INFO - Anticipated number of merges: 8
2026-03-16 14:13:41,976 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 156
2026-03-16 14:13:41,986 - INFO - Dataset 1 shape: (66476, 8), unique PatientIDs: 66476
2026-03-16 14:13:41,995 - INFO - Dataset 2 shape: (66476, 6), unique PatientIDs: 66476
2026-03-16 14:13:42,003 - INFO - Dataset 3 shape: (66476, 11), unique PatientIDs: 66476
2026-03-16 14:13:42,010 - INFO - Dataset 4 shape: (66476, 4), unique PatientIDs: 66476
2026-03-16 14:13:42,018 - INFO - Dataset 5 shape: (66476, 8), unique PatientIDs: 66476
2026-03-16 14:13:42,025 - INFO - Dataset 6 shape: (66476, 76), unique PatientIDs: 66476
2026-03-16 14:13:42,033 - INFO - Dataset 7 shape: (66476, 9), unique PatientIDs: 66476
2026-03-16 14:13:42,038 - INFO - Dataset 8 shape: (66476, 39), unique PatientIDs: 66476
2026-03-16 14:13:42,043 - INFO - Dataset 9 shape: (66187, 3), unique PatientIDs: 66187
2026-03-

In [45]:
training_df.shape

(66187, 156)

## Export dataframe

In [46]:
training_df.to_csv('../outputs/1L_features_training.csv', index = False)

In [47]:
# Save dtypes
training_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_training_dtypes.csv')